# Week 4: Segmentation — Chest X-Ray Pneumonia

**Goal:** Train a U-Net to segment pneumonia-affected regions in chest X-rays.

Pipeline: COCO bounding boxes → SAM (precise masks) → U-Net (segmentation) → evaluation & visualization.

---
## Step 0: Upload your data zip

In [ ]:
from google.colab import files
import zipfile
import os
import json
import shutil
from pathlib import Path
import random

random.seed(42)
DATA_DIR = Path('dataset')
OUT_DIR = Path('seg_data')
MODEL_DIR = Path('trained_model')
PRED_DIR = Path('predictions')
for d in [OUT_DIR, MODEL_DIR, PRED_DIR]:
    d.mkdir(exist_ok=True)

print('Please upload colab_package.zip')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'Extracting {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('dataset_raw')
print('Extracted.')
os.system('ls -lh dataset_raw/')

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics segmentation-models-pytorch torch torchvision
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python-headless

# Download SAM ViT-B weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O sam_vit_b.pth
print('Dependencies installed.')

## Step 2: Convert COCO → YOLO & generate masks (split train/val)

Each image is split into train/val. Initial masks are rectangles derived from bounding boxes — we'll refine with SAM in Step 3.

In [ ]:
import numpy as np
from PIL import Image

RAW = Path('dataset_raw')
TRAIN_RATIO = 0.8

with open(RAW / 'annotations_coco.json') as f:
    coco = json.load(f)

image_info = {img['id']: img for img in coco['images']}
anns_by_img = {}
for ann in coco['annotations']:
    anns_by_img.setdefault(ann['image_id'], []).append(ann)

img_ids = list(anns_by_img.keys())
random.shuffle(img_ids)
split = int(len(img_ids) * TRAIN_RATIO)
train_ids = set(img_ids[:split])
val_ids = set(img_ids[split:])
print(f'Train: {len(train_ids)}, Val: {len(val_ids)}')

for split_name, split_ids in [('train', train_ids), ('val', val_ids)]:
    (OUT_DIR / 'images' / split_name).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / 'masks' / split_name).mkdir(parents=True, exist_ok=True)

    for img_id in split_ids:
        img = image_info[img_id]
        fname = img['file_name']
        w, h = img['width'], img['height']

        src = RAW / 'images' / fname
        dst = OUT_DIR / 'images' / split_name / fname
        shutil.copy2(src, dst)

        mask = np.zeros((h, w), dtype=np.uint8)
        for ann in anns_by_img[img_id]:
            bx, by, bw, bh = ann['bbox']
            x1, y1 = max(0, int(bx)), max(0, int(by))
            x2, y2 = min(w, int(bx + bw)), min(h, int(by + bh))
            if x2 > x1 and y2 > y1:
                mask[y1:y2, x1:x2] = 255

        Image.fromarray(mask).save(OUT_DIR / 'masks' / split_name / (Path(fname).stem + '.png'))

print('Initial rectangular masks created.')
print('Next: refine with SAM for precise organic shapes.')

## Step 3: Refine masks with SAM

For each image, prompt SAM with the bounding box → get a precise, organically-shaped mask. Replaces the rectangular mask with the SAM output.

In [ ]:
import torch
from segment_anything import sam_model_registry, SamPredictor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b.pth')
sam.to(device=DEVICE)
predictor = SamPredictor(sam)
print('SAM loaded.')

with open(RAW / 'annotations_coco.json') as f:
    coco = json.load(f)
image_info = {img['id']: img for img in coco['images']}
anns_by_img = {}
for ann in coco['annotations']:
    anns_by_img.setdefault(ann['image_id'], []).append(ann)

for split_name, split_ids in [('train', train_ids), ('val', val_ids)]:
    img_dir = OUT_DIR / 'images' / split_name
    mask_dir = OUT_DIR / 'masks' / split_name
    for img_path in sorted(img_dir.iterdir()):
        # Find the corresponding image id
        img_id = None
        for iid, info in image_info.items():
            if Path(info['file_name']).stem == img_path.stem:
                img_id = iid
                break
        if img_id is None or img_id not in anns_by_img:
            continue

        pil = Image.open(img_path).convert('RGB')
        img_np = np.array(pil)
        predictor.set_image(img_np)

        merged_mask = np.zeros(img_np.shape[:2], dtype=np.uint8)
        for ann in anns_by_img[img_id]:
            bx, by, bw, bh = ann['bbox']
            box = np.array([bx, by, bx + bw, by + bh])
            masks, scores, _ = predictor.predict(
                point_coords=None,
                point_labels=None,
                box=box[None, :],
                multimask_output=False,
            )
            if len(masks) > 0:
                merged_mask = np.maximum(merged_mask, (masks[0] * 255).astype(np.uint8))

        Image.fromarray(merged_mask).save(mask_dir / (img_path.stem + '.png'))
    print(f'{split_name}: SAM masks generated for {len(list(img_dir.iterdir()))} images')

print('\nAll masks refined with SAM.')

## Step 4: Train U-Net

In [ ]:
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from PIL import Image
import torchvision.transforms.functional as TF

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

class SegDataset(Dataset):
    def __init__(self, images_dir, masks_dir, augment=True):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.augment = augment
        self.images = sorted([f for f in images_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        mask = Image.open(self.masks_dir / (self.images[idx].stem + '.png')).convert('L')
        img = TF.resize(img, [256, 256])
        mask = TF.resize(mask, [256, 256], interpolation=TF.InterpolationMode.NEAREST)
        img = TF.to_tensor(img)
        mask = TF.to_tensor(mask)
        if mask.max() > 0:
            mask = mask / mask.max()
        if self.augment:
            if random.random() < 0.5:
                img = TF.hflip(img); mask = TF.hflip(mask)
            if random.random() < 0.3:
                img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))
        return img, mask

def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def combined_loss(pred, target):
    bce = F.binary_cross_entropy_with_logits(pred, target)
    return 0.5 * bce + 0.5 * dice_loss(pred, target)

def compute_iou_dice(pred_logits, target, threshold=0.5):
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    iou = (intersection + 1e-7) / (union + 1e-7)
    dice = (2 * intersection + 1e-7) / (pred.sum() + target.sum() + 1e-7)
    return float(iou), float(dice)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

train_ds = SegDataset(OUT_DIR / 'images' / 'train', OUT_DIR / 'masks' / 'train', augment=True)
val_ds = SegDataset(OUT_DIR / 'images' / 'val', OUT_DIR / 'masks' / 'val', augment=False)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)

model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=1).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
use_amp = DEVICE == 'cuda'
scaler = GradScaler(enabled=use_amp)

best_dice = -1
best_path = MODEL_DIR / 'best_model.pth'
patience_counter = 0
patience = 20

for epoch in range(1, 101):
    model.train()
    train_loss = 0
    for img, mask in train_loader:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        if use_amp:
            with torch.amp.autocast('cuda'):
                logits = model(img)
                loss = combined_loss(logits, mask)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(img)
            loss = combined_loss(logits, mask)
            loss.backward()
            optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0; iou_sum = 0; dice_sum = 0; n = 0
    with torch.no_grad():
        for img, mask in val_loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)
            logits = model(img)
            val_loss += combined_loss(logits, mask).item()
            iou, dice = compute_iou_dice(logits, mask)
            iou_sum += iou; dice_sum += dice; n += 1
    avg_train = train_loss / max(1, len(train_loader))
    avg_val = val_loss / max(1, len(val_loader))
    avg_iou = iou_sum / max(1, n)
    avg_dice = dice_sum / max(1, n)
    scheduler.step(avg_val)
    print(f'Epoch {epoch:3d} | train_loss: {avg_train:.4f} | val_loss: {avg_val:.4f} | mIoU: {avg_iou:.4f} | Dice: {avg_dice:.4f}')

    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), best_path)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}.')
            break

print(f'\nBest Dice: {best_dice:.4f}')

## Step 5: Evaluate & visualize

In [ ]:
import json
from PIL import Image, ImageDraw, ImageFont

model = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pth', map_location=DEVICE))
model.eval()

iou_sum = 0; dice_sum = 0; acc_sum = 0
tp = fp = fn = tn = 0; n = 0
with torch.no_grad():
    for img, mask in val_loader:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        logits = model(img)
        iou, dice = compute_iou_dice(logits, mask)
        pred = (torch.sigmoid(logits) > 0.5).float()
        acc = float((pred == mask).float().sum() / torch.numel(pred))
        iou_sum += iou; dice_sum += dice; acc_sum += acc; n += 1
        tp += float(((pred == 1) & (mask == 1)).sum())
        fp += float(((pred == 1) & (mask == 0)).sum())
        fn += float(((pred == 0) & (mask == 1)).sum())
        tn += float(((pred == 0) & (mask == 0)).sum())

results = {
    'mIoU': iou_sum / n,
    'dice': dice_sum / n,
    'pixel_accuracy': acc_sum / n,
    'precision': tp / (tp + fp + 1e-7),
    'recall': tp / (tp + fn + 1e-7),
}
print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(f'  mIoU:           {results["mIoU"]:.4f}')
print(f'  Dice:           {results["dice"]:.4f}')
print(f'  Pixel Accuracy: {results["pixel_accuracy"]:.4f}')
print(f'  Precision:      {results["precision"]:.4f}')
print(f'  Recall:         {results["recall"]:.4f}')

with open(MODEL_DIR / 'metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

# Visualization
def overlay_mask(pil_img, mask, color, alpha=0.4):
    img = pil_img.convert('RGB').copy()
    img_arr = np.array(img).astype(np.float32)
    mask_arr = mask > 127
    color_arr = np.array(color, dtype=np.float32)
    img_arr[mask_arr] = alpha * color_arr + (1 - alpha) * img_arr[mask_arr]
    return Image.fromarray(img_arr.astype(np.uint8))

def add_text(img, text, position=(5, 5)):
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 16)
    except (OSError, IOError):
        font = ImageFont.load_default()
    bbox = draw.textbbox(position, text, font=font)
    draw.rectangle(bbox, fill=(0, 0, 0))
    draw.text(position, text, fill=(255, 255, 255), font=font)
    return img

val_img_dir = OUT_DIR / 'images' / 'val'
val_mask_dir = OUT_DIR / 'masks' / 'val'
for img_path in sorted(val_img_dir.iterdir()):
    pil = Image.open(img_path).convert('RGB')
    gt_mask = np.array(Image.open(val_mask_dir / (img_path.stem + '.png')).convert('L'))
    gt_mask = (gt_mask > 127).astype(np.uint8)
    pil_resized = pil.resize((256, 256))
    img_tensor = TF.to_tensor(pil_resized).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(img_tensor)
        pred = (torch.sigmoid(logits) > 0.5).cpu().squeeze().numpy().astype(np.uint8)
    gt_overlay = add_text(overlay_mask(pil_resized, gt_mask, (0, 255, 0)), 'Ground Truth')
    pred_overlay = add_text(overlay_mask(pil_resized, pred, (255, 0, 0)), 'Prediction')
    composite = Image.new('RGB', (256 * 2 + 10, 256), (255, 255, 255))
    composite.paste(gt_overlay, (0, 0))
    composite.paste(pred_overlay, (266, 0))
    composite.save(PRED_DIR / f'pred_{img_path.stem}.png')
print(f'Saved {len(list(val_img_dir.iterdir()))} prediction images to {PRED_DIR}/')

## Step 6: Download results

In [ ]:
import zipfile

with zipfile.ZipFile('week4_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in MODEL_DIR.rglob('*'):
        if f.is_file():
            zf.write(f, str(f.relative_to('.')))
    for f in PRED_DIR.rglob('*'):
        if f.is_file():
            zf.write(f, str(f.relative_to('.')))

print(f'Created week4_results.zip ({os.path.getsize("week4_results.zip") / 1024:.1f} KB)')
from google.colab import files
files.download('week4_results.zip')

## Done!

Downloaded zip contains:
- `trained_model/best_model.pth` — trained U-Net
- `trained_model/metrics.json` — mIoU, Dice, accuracy
- `predictions/` — side-by-side ground truth vs prediction overlays